Bronze Bootstrap Ingestion

Objective
Ingest historical USGS earthquake data and initialize the Silver table with a consistent schema.

Process Overview

Retrieve 30 days of earthquake data from the USGS API and store raw JSON in the Bronze layer (/Files/bronze_usgs_30days_{timestamp}.json).
Load the raw JSON file into a DataFrame for processing.
Flatten the nested JSON structure by extracting fields from features, properties, and geometry.
Convert event timestamps from epoch milliseconds to UTC format and standardize column names.
Write the transformed dataset to the earthquakes_silver Delta table as the initial bootstrap load.

This bootstrap process establishes the baseline dataset for subsequent incremental ingestion and merging.

In [1]:
# usgs api to raw bronze ingestion
# Lakehouse
#  ├── Files
#  │     bronze_usgs_30days_YYYYMMDD_HHMMSS.json   (raw API response)
#  │
#  └── Tables
#        (none yet for this step)
#
# Notes
#  - This step pulls the last 30 days of USGS earthquakes via REST API
#  - The raw payload is stored as JSON in Files to preserve source fidelity
#  - No transformations here (Bronze landing zone)

import requests
import datetime
#print(dir(datetime))

# Define 30-day UTC window
end = datetime.datetime.now(datetime.timezone.utc)
start = end - datetime.timedelta(days=30)

# USGS API request
#API document From https://earthquake.usgs.gov/fdsnws/event/1/;#parameters
#url: https://earthquake.usgs.gov/fdsnws/event/1/[METHOD[?PARAMETERS]]
url = "https://earthquake.usgs.gov/fdsnws/event/1/query"
params = {
    "format": "geojson",
    "starttime": start.strftime("%Y-%m-%dT%H:%M:%S"),
    "endtime": end.strftime("%Y-%m-%dT%H:%M:%S")
}

response = requests.get(url, params=params, timeout=30)
response.raise_for_status()

# Write raw JSON to Bronze layer
timestamp = end.strftime("%Y%m%d_%H%M%S")
# Ensure folder exists (safe even if already exists)
notebookutils.fs.mkdirs("Files/bronze_usgs_30days_base")

file_path = f"Files/bronze_usgs_30days_base/bronze_usgs_30days_{timestamp}.json"
#mssparkutils.fs.put(file_path, response.text, True)
notebookutils.fs.put(file_path, response.text, True)
print(f"Done Ingesting raw file: {file_path}")

StatementMeta(, 0b27427b-f230-4c99-9e4d-bf9cb685e323, 3, Finished, Available, Finished, False)

Done Ingesting raw file: Files/bronze_usgs_30days_base/bronze_usgs_30days_20260320_210454.json


In [3]:
# ------------------------------------------------------------
# BRONZE → SILVER TRANSFORMATION (USGS GeoJSON FeatureCollection)
#
# Goal
#   Read the raw USGS GeoJSON response (FeatureCollection) from Files,
#   analyze its structure, then transform it into a clean Silver Delta table.
#
# Lakehouse Structure
#
# Lakehouse
# ├── Files
# │     bronze_usgs_30days_20260305_145844.json     (raw API payload)
# │
# └── Tables
#       earthquakes_silver                          (clean structured dataset)
#       earthquakes_gold                            (aggregated analytics - later)
#
# Processing Steps (Informative + controlled)
#   1. Read Bronze JSON as a FeatureCollection (multiline JSON)
#   2. Inspect schema and confirm top level structure (metadata + features[])
#   3. Validate record counts (1 wrapper row, N features)
#   4. Explode features[] into event level rows
#   5. Extract and flatten fields from properties and geometry.coordinates
#   6. Clean and standardize columns (naming + types)
#   7. Write the dataset as a Silver Delta table
#   8. Verify the Silver table (row count + sample rows)
#
# Output
#   Tables/earthquakes_silver
#
# Notes
#   - Bronze stores the raw API response exactly as returned (FeatureCollection).
#   - Silver is a structured, query friendly Delta table (Parquet under the hood).
# ------------------------------------------------------------

from pyspark.sql.functions import col, explode, size, to_timestamp, from_unixtime

# -----------------------------
# Step 1 — Read Bronze JSON (FeatureCollection)
# -----------------------------
bronze_path = "Files/bronze_usgs_30days_base/bronze_*.json"

# important for FeatureCollection JSON
df_fc = (spark.read.option("multiline", "true").json(bronze_path))

# -----------------------------
# Step 2 — Inspect schema (verify metadata + features[])
# -----------------------------
#df_fc.printSchema()

# -----------------------------
# Step 3 — Validate structure (wrapper row count + features count)
# -----------------------------
print("Wrapper row count (should be 1):", df_fc.count())

df_fc.select(
    col("type").alias("root_type"),
    col("metadata.count").alias("metadata_count"),
    size(col("features")).alias("features_array_size")
).show(truncate=False)

# Step 4 — Explode features[] into event level rows
# -----------------------------
df_features = df_fc.select(explode("features").alias("feature"))

# Optional: inspect the schema after exploding
#df_features.printSchema()

# -----------------------------
# Step 5 — Extract and flatten (properties + geometry)
# -----------------------------
df_flat = df_features.select(
    col("feature.id").alias("event_id"),
    col("feature.type").alias("feature_type"),
    col("feature.properties.mag").alias("magnitude"),
    col("feature.properties.place").alias("place"),
    col("feature.properties.time").alias("event_time_ms"),
    col("feature.properties.updated").alias("updated_time_ms"),
    col("feature.properties.status").alias("status"),
    col("feature.properties.magType").alias("mag_type"),
    col("feature.properties.net").alias("net"),
    col("feature.properties.code").alias("code"),
    col("feature.properties.url").alias("url"),
    col("feature.geometry.type").alias("geometry_type"),
    col("feature.geometry.coordinates")[0].alias("longitude"),
    col("feature.geometry.coordinates")[1].alias("latitude"),
    col("feature.geometry.coordinates")[2].alias("depth_km")
)
# -----------------------------
# Step 6 — Clean / standardize types (timestamps)
#   USGS time fields are epoch milliseconds
# -----------------------------
df_silver = df_flat.select(
    "*",
    to_timestamp(from_unixtime(col("event_time_ms")/1000)).alias("event_time_utc"),
    to_timestamp(from_unixtime(col("updated_time_ms")/1000)).alias("updated_time_utc")
).drop("event_time_ms", "updated_time_ms")

# Quick sanity check
#display(df_silver.limit(10))
# -----------------------------
# Step 7 — Write Silver Delta table
# -----------------------------
(
    df_silver.write.format("delta").mode("overwrite").saveAsTable("earthquakes_silver")
)

# -----------------------------
# Step 8 — Verify Silver table
# -----------------------------
print("Silver row count:", spark.table("earthquakes_silver").count())
display(spark.table("earthquakes_silver").limit(10))

# -----------------------------
# Step 9 — Store headers to be used as schema for new hourly files
# -----------------------------
silver_cols= df_silver.columns
print(silver_cols)
display(df_silver.limit(5))

StatementMeta(, 0b27427b-f230-4c99-9e4d-bf9cb685e323, 5, Finished, Available, Finished, False)

Wrapper row count (should be 1): 1
+-----------------+--------------+-------------------+
|root_type        |metadata_count|features_array_size|
+-----------------+--------------+-------------------+
|FeatureCollection|10708         |10708              |
+-----------------+--------------+-------------------+

Silver row count: 10708


SynapseWidget(Synapse.DataFrame, f2fb3473-ddb2-472c-9e96-f68e1a73ca8b)

['event_id', 'feature_type', 'magnitude', 'place', 'status', 'mag_type', 'net', 'code', 'url', 'geometry_type', 'longitude', 'latitude', 'depth_km', 'event_time_utc', 'updated_time_utc']


SynapseWidget(Synapse.DataFrame, 3fedde80-ef6d-4585-8a34-330a53f515bd)